# 🔬 SCUA Concept Explanation Pipeline (GPQA Dataset) - Chain of Thought (CoT 600w)
### Model: `deepseek-v4-flash` | High-Concurrency Batch Engine (500 RPM / 4M TPM)

This notebook implements the **SCUA (Scientific Concept Understanding)** workflow configured for the **Extended Chain of Thought (CoT) Baseline (max 600 words)** for scientific multiple-choice questions from the **GPQA** dataset.

---
### 🛠️ Architecture & Features
1. **Extended Concept Chain-of-Thought (CoT 600w):** Generates thorough step-by-step conceptual mechanism explanations without analogies (`MAX_TOKENS = 1024`).
2. **Word-Limited Concept Extraction:** Strict 2–6 word concept phrasing.
3. **Timestamped Runs:** Each execution creates a dedicated folder `results/run_YYYY-MM-DD_HH-MM-SS/`.
4. **High Concurrency:** Multi-threaded execution (`ThreadPoolExecutor`) with 15 workers.
5. **Thinking Mode Disabled:** Fast direct output without internal CoT tokens.
6. **Defect Tracking & Labeling:** Flags defective rows (`"is_defective": True/False`, `"defect_reason"`).
7. **CSV & JSONL Storage:** Saves full JSONL, full CSV, and clean (defect-free) CSV files.
8. **Customizable Sample Viewer:** Inspect any optional number of generated samples with clean formatting.

In [72]:
import os
import json
import re
import time
import random
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, Any, Optional, List, Set
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from huggingface_hub import login

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


## 1. ⚙️ Configuration & Timestamped Run Setup (CoT 600 Words)
Sets up API access, CoT 600w hyperparameters (`MAX_TOKENS = 1024`), and creates a new timestamped results directory.

In [73]:
# ==============================================================================
# CONFIGURATION & TIMESTAMPED RUN SETUP
# ==============================================================================

# API Configuration
AVALAI_API_BASE = "https://api.avalai.ir/v1"
AVALAI_API_KEY = os.environ.get("AVALAI_API_KEY", "")
TEACHER_MODEL = "deepseek-v4-flash"

# Hugging Face Login
try:
    if os.environ.get("HF_TOKEN"):
        login(os.environ.get("HF_TOKEN"))
except Exception as e:
    print(f"HF Login note: {e}")

# Dataset Configuration (Input from SCUA-main)
DATASET_NAME = "GPQA"
INPUT_DATASET_PATH = f"SCUA-main/dataset/{DATASET_NAME}/{DATASET_NAME}_dataset.jsonl"

# Timestamped Results Directory (Outside SCUA-main)
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
RESULTS_DIR = Path(f"results/run_{RUN_TIMESTAMP}")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Output File Paths (Labeled with cot-600w for distinct thesis tracking)
EXPERIMENT_CONDITION_LABEL = "cot-600w"
OUTPUT_DATASET_PATH = str(RESULTS_DIR / f"{DATASET_NAME}_{EXPERIMENT_CONDITION_LABEL}_{TEACHER_MODEL}.jsonl")
OUTPUT_CSV_PATH = str(RESULTS_DIR / f"{DATASET_NAME}_{EXPERIMENT_CONDITION_LABEL}_{TEACHER_MODEL}.csv")
OUTPUT_CLEAN_CSV_PATH = str(RESULTS_DIR / f"{DATASET_NAME}_{EXPERIMENT_CONDITION_LABEL}_{TEACHER_MODEL}_clean.csv")

# Concurrency Tuning (AvalAI Tier 2: 500 RPM / 4M TPM)
MAX_WORKERS = 15          # 15 concurrent threads

# Hyperparameters for 600-Word Chain of Thought
TEMPERATURE = 0.0
MAX_TOKENS = 1024         # 1024 tokens to comfortably accommodate up to 600 words (~800-850 tokens)
MAX_RETRIES = 5
RETRY_DELAY = 2           # seconds

# Initialize OpenAI-compatible client
client = OpenAI(
    api_key=AVALAI_API_KEY,
    base_url=AVALAI_API_BASE
)

print(f"📁 Results Directory: {RESULTS_DIR.resolve()}")
print(f"🔗 Endpoint:          {AVALAI_API_BASE}")
print(f"🤖 Model:             {TEACHER_MODEL}")
print(f"🧠 Experiment Mode:   Extended Chain of Thought (CoT 600w / Max Tokens: {MAX_TOKENS})")
print(f"⚡ Concurrency:       {MAX_WORKERS} workers")
print(f"🚫 Thinking:          Disabled")
print(f"📂 Input Path:        {INPUT_DATASET_PATH}")
print(f"💾 Output JSONL:      {OUTPUT_DATASET_PATH}")
print(f"📊 Output CSV:        {OUTPUT_CSV_PATH}")

📁 Results Directory: /Users/neo/Downloads/🧩Analogy/Thesis Runs/🐋think enable disable/🪼generating ananlogies and cots via deepseek /results/run_2026-08-11_10-37-04
🔗 Endpoint:          https://api.avalai.ir/v1
🤖 Model:             deepseek-v4-flash
🧠 Experiment Mode:   Extended Chain of Thought (CoT 600w / Max Tokens: 1024)
⚡ Concurrency:       15 workers
🚫 Thinking:          Disabled
📂 Input Path:        SCUA-main/dataset/GPQA/GPQA_dataset.jsonl
💾 Output JSONL:      results/run_2026-08-11_10-37-04/GPQA_cot-600w_deepseek-v4-flash.jsonl
📊 Output CSV:        results/run_2026-08-11_10-37-04/GPQA_cot-600w_deepseek-v4-flash.csv


## 2. 📝 Flexible Prompt Registry
Manage and customize prompt formats here. Includes the Extended Chain of Thought (CoT 600w) baseline prompt.

In [74]:
# ==============================================================================
# PROMPT REGISTRY: Define and customize prompt formats here
# ==============================================================================

PROMPT_REGISTRY = {
    # --------------------------------------------------------------------------
    # 1. CONCEPT EXTRACTION PROMPTS
    # --------------------------------------------------------------------------
    "concept_extraction": {
        # Word-limited concise format (strictly 2 to 6 words maximum)
        "concise_word_limited": """Given a scientific question, identify the single core scientific concept, law, reaction, or principle required to solve it.

Requirements:
1. The concept must be a concise title/phrase (strictly between 2 and 6 words maximum).
2. Do NOT summarize the question, do NOT describe steps, and do NOT write full sentences.
3. Output ONLY a valid, parsable JSON object.

Example outputs:
{{"key_scientific_concept": "Corey-Chaykovsky Epoxidation"}}
{{"key_scientific_concept": "Energy-Time Uncertainty Principle"}}
{{"key_scientific_concept": "Poincaré Disk Hyperbolic Metric"}}

This is the scientific question:
{question}

The key scientific concept:""",

        # Standard SCUA format (unconstrained length)
        "scua_default": """Given a scientific question, you should show the key scientific concept related to this scientific question.
This is a scientific question:
{question}
You should only output in a parsible JSON format. The example outputs look like:

{{"key_scientific_concept": "The_key_scientific_concept"}}

The key scientific concept:"""
    },

    # --------------------------------------------------------------------------
    # 2. CONCEPT EXPLANATION & ANALOGY PROMPTS
    # --------------------------------------------------------------------------
    "free_form_analogy": {
        # Extended Chain of Thought Baseline (max 600 words without analogies)
        "cot_600_words": """Please provide a concise, conceptually informative explanation of the following academic concept in no more than 600 words.

Concept: {concept}

Requirements:

1. Explain the concept at a broad, introductory level, focusing on its general meaning, central idea, and overall importance within its field.

2. Use discipline-appropriate language, but avoid highly specific technical details that could function as clues to a particular test question.

3. Focus on general conceptual understanding rather than equations, formulas, calculations, numerical values, thresholds, named laws, named theories, specific classifications, diagnostic criteria, or detailed derivations.

4. Do not list or separately identify specific mechanisms, components, stages, pathways, variables, conditions, exceptions, characteristics, or consequences when those details could distinguish between closely related alternatives.

5. Avoid stating relationships, contrasts, modifications, causes, effects, or defining features with enough specificity that they could directly determine the answer to a multiple-choice question.

6. When the concept contains several subcomponents or processes, describe them only at a high level rather than enumerating or defining them individually.

7. Do NOT use analogies, metaphors, worked examples, case examples, or hypothetical scenarios.

8. Do not provide problem-solving steps, calculations, decision rules, elimination strategies, or instructions for applying the concept to a specific question.

9. Do not infer, reconstruct, mention, or discuss any unseen question, answer choices, correct answer, or likely assessment context.

10. The explanation must be self-contained and educational, but intentionally remain at a general conceptual level.

11. Keep the explanation strictly under 600 words.

Explanation:
""",

        # Chain of Thought Baseline (max 300 words without analogies)
        "cot_300_words": """Please provide a clear, step-by-step chain of thought explanation with no more than 300 words to explain the core mechanisms and principles of the scientific concept: {concept}

Requirements:
1. Explain the underlying principles, logical steps, and scientific mechanisms directly.
2. Do NOT use analogies or metaphors.
3. Keep the total explanation strictly under 300 words.

Chain of Thought:""",

        # Single 600-word analogy format
        "free_form_600_words": """Please use an analogy with no more than 600 words to explain the scientific concept: {concept}
Analogy""",

        # 2 Distinct Analogies (300 words each)
        "two_analogies_300w_each": """Please provide 2 distinct and separate analogies from two different everyday domains to explain the scientific concept: {concept}

Requirements:
1. Each analogy must be completely distinct from the other (using different scenarios/mechanisms).
2. Each individual analogy must be no more than 300 words (maximum 600 words total).
3. Clearly format your output with "Analogy 1:" and "Analogy 2:".

Analogies:""",

        # 3 Distinct Analogies (200 words each / 600 words total)
        "three_analogies_200w_each": """Please provide 3 distinct and separate analogies from three different everyday domains to explain the scientific concept: {concept}

Requirements:
1. Each analogy must be completely distinct from the others (using different everyday domains/mechanisms).
2. Each individual analogy must be no more than 200 words (maximum 600 words total).
3. Clearly format your output with "Analogy 1:", "Analogy 2:", and "Analogy 3:".

Analogies:""",

        # Standard SCUA single 300-word analogy format
        "scua_default": """Please use an analogy with no more than 300 words to explain the scientific concept: {concept}
Analogy"""
    }
}

# Select active prompt formats:
ACTIVE_CONCEPT_PROMPT_KEY = "concise_word_limited"   # 2-6 words limited concept extraction
ACTIVE_ANALOGY_PROMPT_KEY = "cot_600_words"          # Extended Chain of Thought (CoT 600w)

print(f"🎯 Active Concept Prompt: '{ACTIVE_CONCEPT_PROMPT_KEY}'")
print(f"🎯 Active Explanation Prompt: '{ACTIVE_ANALOGY_PROMPT_KEY}'")

🎯 Active Concept Prompt: 'concise_word_limited'
🎯 Active Explanation Prompt: 'cot_600_words'


## 3. 🛠️ Helper & Generation Functions
Functions with thinking mode disabled, retry handling, and non-empty response validation.

In [75]:
def call_llm(
    messages: List[Dict[str, str]],
    model: str = TEACHER_MODEL,
    temperature: float = TEMPERATURE,
    max_tokens: int = MAX_TOKENS,
    max_retries: int = MAX_RETRIES,
    retry_delay: int = RETRY_DELAY
) -> str:
    """
    Calls the OpenAI-compatible API with thinking mode explicitly disabled
    and exponential backoff retry logic.
    """
    for attempt in range(1, max_retries + 1):
        try:
            # Disabling thinking mode on DeepSeek / AvalAI endpoints
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
                extra_body={
                    "thinking": {"type": "disabled"}
                }
            )
            choice = response.choices[0]
            content = choice.message.content or ""
            
            # Strip <think> tags if any remain
            clean_content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
            if clean_content:
                return clean_content
            
            # If message.content was empty, check reasoning_content fallback
            if hasattr(choice.message, "reasoning_content") and choice.message.reasoning_content:
                return choice.message.reasoning_content.strip()
        except Exception as e:
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens
                )
                choice = response.choices[0]
                content = choice.message.content or ""
                clean_content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
                if clean_content:
                    return clean_content
            except Exception as e2:
                pass
                
            if attempt == max_retries:
                return ""
                
        time.sleep(retry_delay * (2 ** (attempt - 1)))
    return ""


def parse_extracted_concept(raw_output: str) -> str:
    """
    Extracts the clean concept string from JSON, markdown code blocks, or raw text.
    """
    if not raw_output:
        return ""
        
    # 1. Direct JSON parsing
    try:
        data = json.loads(raw_output)
        if isinstance(data, dict) and "key_scientific_concept" in data:
            return str(data["key_scientific_concept"]).strip()
    except Exception:
        pass

    # 2. Regex search for key_scientific_concept
    json_match = re.search(r'"key_scientific_concept"\s*:\s*"([^"]+)"', raw_output, re.DOTALL)
    if json_match:
        return json_match.group(1).strip()

    # 3. Markdown code block extraction ```json ... ```
    code_block_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw_output, re.DOTALL)
    if code_block_match:
        try:
            data = json.loads(code_block_match.group(1))
            if "key_scientific_concept" in data:
                return str(data["key_scientific_concept"]).strip()
        except Exception:
            pass

    # 4. Fallback: Return raw text with quotes stripped
    return raw_output.strip(" \t\n\r\"'{}[]`")


def extract_concept(
    question_stem: str,
    prompt_key: str = ACTIVE_CONCEPT_PROMPT_KEY,
    model: str = TEACHER_MODEL,
    max_attempts: int = 3
) -> Dict[str, str]:
    """Generates and parses the key scientific concept for a question stem."""
    template = PROMPT_REGISTRY["concept_extraction"][prompt_key]
    prompt = template.format(question=question_stem)
    messages = [{"role": "user", "content": prompt}]
    
    raw_response = ""
    clean_concept = ""
    for _ in range(max_attempts):
        raw_response = call_llm(messages, model=model)
        clean_concept = parse_extracted_concept(raw_response)
        if clean_concept:
            break
        time.sleep(1)
        
    return {
        "raw_response": raw_response,
        "clean_concept": clean_concept
    }


def generate_free_form_analogy(
    concept: str,
    prompt_key: str = ACTIVE_ANALOGY_PROMPT_KEY,
    model: str = TEACHER_MODEL,
    max_attempts: int = 3
) -> str:
    """Generates and validates an extended Chain of Thought explanation for the given scientific concept."""
    if not concept:
        return ""
        
    template = PROMPT_REGISTRY["free_form_analogy"][prompt_key]
    prompt = template.format(concept=concept)
    messages = [{"role": "user", "content": prompt}]
    
    explanation = ""
    for _ in range(max_attempts):
        explanation = call_llm(messages, model=model)
        if explanation and "didn't specify" not in explanation.lower() and "missing the specific" not in explanation.lower():
            return explanation
        time.sleep(1)
        
    return explanation

print("✅ Helper functions loaded successfully.")

✅ Helper functions loaded successfully.


## 4. 🧪 Single-Sample Interactive Test (CoT 600w)
Run a quick test on a single question from GPQA to preview the extended Chain of Thought explanation.

In [76]:
# Single sample test with a question from GPQA
sample_question = (
    "trans-cinnamaldehyde was treated with methylmagnesium bromide, forming product 1.\n\n"
    "1 was treated with pyridinium chlorochromate, forming product 2.\n\n"
    "3 was treated with (dimethyl(oxo)-l6-sulfaneylidene)methane in DMSO at elevated temperature, forming product 3.\n\n"
    "how many carbon atoms are there in product 3?"
)

print(f"📌 Sample Question:\n{sample_question}\n" + "-"*60)

# 1. Extract Concise Concept
concept_result = extract_concept(sample_question)
print(f"💡 Raw Model Output:\n{concept_result['raw_response']}")
print(f"✨ Concise Concept (2-6 words):  {concept_result['clean_concept']}\n" + "-"*60)

# 2. Generate Extended Chain of Thought Explanation (CoT 600w)
cot_result = generate_free_form_analogy(concept_result['clean_concept'])
total_words = len(cot_result.split())
print(f"🧠 Generated Chain of Thought ({total_words} words):\n{cot_result}\n" + "="*60)

📌 Sample Question:
trans-cinnamaldehyde was treated with methylmagnesium bromide, forming product 1.

1 was treated with pyridinium chlorochromate, forming product 2.

3 was treated with (dimethyl(oxo)-l6-sulfaneylidene)methane in DMSO at elevated temperature, forming product 3.

how many carbon atoms are there in product 3?
------------------------------------------------------------
💡 Raw Model Output:
{"key_scientific_concept": "Grignard Reaction"}
✨ Concise Concept (2-6 words):  Grignard Reaction
------------------------------------------------------------
🧠 Generated Chain of Thought (494 words):
The Grignard reaction is a cornerstone method in organic chemistry for constructing new carbon-carbon bonds. Its central idea is the use of a highly reactive organometallic reagent, known as a Grignard reagent, to attach a carbon-based group to a variety of other molecules. This process is fundamental because it allows chemists to build complex, carbon-rich structures from simpler buildin

## 5. ⚡ Concurrent Batch Pipeline with Defect Labeling
Defines the multithreaded worker and batch processor with real-time defect labeling and thread-safe streaming into the timestamped directory.

In [77]:
def load_jsonl(file_path: str) -> List[Dict[str, Any]]:
    records = []
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        records.append(json.loads(line.strip()))
                    except Exception:
                        pass
    return records


def get_processed_ids(output_path: str) -> Set[str]:
    """Reads existing output JSONL and returns set of successfully processed IDs."""
    processed = set()
    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        rec = json.loads(line.strip())
                        if "id" in rec:
                            processed.add(rec["id"])
                    except Exception:
                        pass
    return processed


def process_single_item(
    item: Dict[str, Any],
    concept_prompt_key: str,
    analogy_prompt_key: str,
    model: str
) -> Dict[str, Any]:
    """
    Processes a single dataset item through Concept Extraction & Explanation Generation,
    adding defect labels and validation status.
    """
    question_stem = item["question"]["stem"]
    
    # 1. Concept Extraction
    concept_res = extract_concept(
        question_stem=question_stem,
        prompt_key=concept_prompt_key,
        model=model
    )
    raw_concept = concept_res["raw_response"]
    clean_concept = concept_res["clean_concept"]
    
    # 2. Concept Explanation / CoT Generation (600w)
    explanation_text = ""
    if clean_concept:
        explanation_text = generate_free_form_analogy(
            concept=clean_concept,
            prompt_key=analogy_prompt_key,
            model=model
        )
    
    # 3. Defect Validation & Labeling
    is_defective = False
    defect_reasons = []
    
    if not clean_concept:
        is_defective = True
        defect_reasons.append("missing_concept")
        
    if not explanation_text:
        is_defective = True
        defect_reasons.append("missing_explanation")
    elif "didn't specify" in explanation_text.lower() or "missing the specific" in explanation_text.lower():
        is_defective = True
        defect_reasons.append("invalid_fallback_explanation")
        
    # Enrich record
    item["key_scientific_concept"] = raw_concept
    item["clean_scientific_concept"] = clean_concept
    item["key_scientific_analogy"] = explanation_text   # Maintained for downstream SCUA compatibility
    
    # Defect & generation metadata
    item["is_defective"] = is_defective
    item["defect_reason"] = ", ".join(defect_reasons) if defect_reasons else None
    item["generation_status"] = "defective" if is_defective else "success"
    item["generation_meta"] = {
        "teacher_model": model,
        "experiment_condition": EXPERIMENT_CONDITION_LABEL,
        "concept_prompt_format": concept_prompt_key,
        "explanation_prompt_format": analogy_prompt_key,
        "thinking_disabled": True,
        "max_tokens": MAX_TOKENS
    }
    
    return item


def run_concurrent_analogy_generation(
    input_path: str = INPUT_DATASET_PATH,
    output_path: str = OUTPUT_DATASET_PATH,
    concept_prompt_key: str = ACTIVE_CONCEPT_PROMPT_KEY,
    analogy_prompt_key: str = ACTIVE_ANALOGY_PROMPT_KEY,
    model: str = TEACHER_MODEL,
    max_workers: int = MAX_WORKERS,
    limit: Optional[int] = None
):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    
    # Load dataset
    dataset = load_jsonl(input_path)
    if limit is not None:
        dataset = dataset[:limit]
        
    total_samples = len(dataset)
    
    # Check already processed IDs for resume support
    processed_ids = get_processed_ids(output_path)
    pending_items = [item for item in dataset if item["id"] not in processed_ids]
    
    print(f"📊 Total Samples in Dataset: {total_samples}")
    print(f"⏩ Already Processed:        {len(processed_ids)}")
    print(f"⏳ Remaining to Process:     {len(pending_items)}")
    print(f"⚡ Concurrency Level:        {max_workers} worker threads")
    print(f"🧠 Experiment Condition:     Extended Chain of Thought (CoT 600w)")
    print(f"💾 Output JSONL Path:        {output_path}")
    
    if not pending_items:
        print("🎉 All samples are already processed!")
        return

    file_lock = threading.Lock()
    success_count = 0
    defective_count = 0

    with tqdm(total=total_samples, initial=len(processed_ids), desc="Generating CoT 600w Explanations") as pbar:
        with open(output_path, "a", encoding="utf-8") as out_f:
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = {
                    executor.submit(
                        process_single_item,
                        item,
                        concept_prompt_key,
                        analogy_prompt_key,
                        model
                    ): item["id"]
                    for item in pending_items
                }
                
                for future in as_completed(futures):
                    item_id = futures[future]
                    try:
                        result_item = future.result()
                        if result_item["is_defective"]:
                            defective_count += 1
                        else:
                            success_count += 1
                            
                        with file_lock:
                            out_f.write(json.dumps(result_item, ensure_ascii=False) + "\n")
                            out_f.flush()
                    except Exception as exc:
                        defective_count += 1
                        print(f"❌ Exception on item {item_id}: {exc}")
                        
                    pbar.update(1)
                    pbar.set_postfix({"Success": success_count, "Defects": defective_count})

    print(f"\n✅ Generation Complete!")
    print(f"   - Successful: {success_count}")
    print(f"   - Defective:  {defective_count}")
    print(f"   - Saved to:   {output_path}")

print("✅ Concurrent batch engine loaded successfully.")

✅ Concurrent batch engine loaded successfully.


## 6. 🚀 Execute Concurrent Batch Generation (CoT 600 Words)
Runs multi-threaded generation with 15 workers. Completes the GPQA dataset in ~1-2 minutes.

In [78]:
# Option A: Quick test on 10 samples
# run_concurrent_analogy_generation(limit=10)

# Option B: Full GPQA dataset (449 samples)
run_concurrent_analogy_generation()

📊 Total Samples in Dataset: 448
⏩ Already Processed:        0
⏳ Remaining to Process:     448
⚡ Concurrency Level:        15 worker threads
🧠 Experiment Condition:     Extended Chain of Thought (CoT 600w)
💾 Output JSONL Path:        results/run_2026-08-11_10-37-04/GPQA_cot-600w_deepseek-v4-flash.jsonl


Generating CoT 600w Explanations: 100%|██████████| 448/448 [08:14<00:00,  1.10s/it, Success=448, Defects=0]


✅ Generation Complete!
   - Successful: 448
   - Defective:  0
   - Saved to:   results/run_2026-08-11_10-37-04/GPQA_cot-600w_deepseek-v4-flash.jsonl


## 7. 🔍 Dataset Quality & Defect Summary
Displays an overall quality summary of the generated dataset.

In [79]:
def inspect_dataset_quality(output_path: str = OUTPUT_DATASET_PATH):
    if not os.path.exists(output_path):
        print(f"Output file not found: {output_path}")
        return
        
    records = load_jsonl(output_path)
    total = len(records)
    defective_records = [r for r in records if r.get("is_defective", False)]
    success_records = [r for r in records if not r.get("is_defective", False)]
    
    print(f"{'='*60}")
    print(f"📊 DATASET QUALITY REPORT (Chain of Thought - CoT 600w)")
    print(f"📁 File: {output_path}")
    print(f"{'='*60}")
    print(f"📌 Total Processed: {total}")
    print(f"✅ Successful:      {len(success_records)} ({len(success_records)/total*100:.1f}%)" if total else "0")
    print(f"⚠️ Defective:       {len(defective_records)} ({len(defective_records)/total*100:.1f}%)" if total else "0")
    print(f"{'='*60}\n")

# Run Quality Summary
inspect_dataset_quality()

📊 DATASET QUALITY REPORT (Chain of Thought - CoT 600w)
📁 File: results/run_2026-08-11_10-37-04/GPQA_cot-600w_deepseek-v4-flash.jsonl
📌 Total Processed: 448
✅ Successful:      448 (100.0%)
⚠️ Defective:       0 (0.0%)



## 8. 📖 Interactive Sample Viewer (Optional Sample Count)
Display and inspect any number of generated samples. You can customize `num_samples`, filter by status (`"all"`, `"success"`, or `"defective"`), or choose random samples.

In [80]:
def display_samples(
    output_path: str = OUTPUT_DATASET_PATH,
    num_samples: int = 5,
    filter_by: str = "all",       # Options: "all", "success", or "defective"
    random_sample: bool = False
):
    """
    Displays formatted samples from the output JSONL file.
    
    Parameters:
        output_path: Path to the generated JSONL file.
        num_samples: Number of samples to display (optional, default: 5).
        filter_by: Filter records by 'all', 'success', or 'defective'.
        random_sample: If True, randomly picks samples; otherwise takes the first N.
    """
    if not os.path.exists(output_path):
        print(f"Output file not found: {output_path}")
        return
        
    records = load_jsonl(output_path)
    if not records:
        print("No records found in output file.")
        return
        
    # Filter records
    if filter_by == "success":
        selected_records = [r for r in records if not r.get("is_defective", False)]
    elif filter_by == "defective":
        selected_records = [r for r in records if r.get("is_defective", False)]
    else:
        selected_records = records
        
    total_available = len(selected_records)
    if total_available == 0:
        print(f"No records found matching filter_by='{filter_by}'.")
        return
        
    # Sample selection
    k = min(num_samples, total_available)
    sample_list = random.sample(selected_records, k) if random_sample else selected_records[:k]
    
    print(f"{'='*70}")
    print(f"📖 DISPLAYING {k} OF {total_available} SAMPLES (Filter: '{filter_by}', Random: {random_sample})")
    print(f"{'='*70}\n")
    
    for i, rec in enumerate(sample_list):
        is_def = rec.get("is_defective", False)
        status_badge = "⚠️ DEFECTIVE" if is_def else "✅ SUCCESS"
        
        print(f"┌{'─'*68}┐")
        print(f"│ SAMPLE #{i+1:02d} | ID: {rec.get('id', 'N/A'):<28} | Status: {status_badge:<11} │")
        print(f"└{'─'*68}┘")
        
        # Question Stem
        q_stem = rec.get("question", {}).get("stem", "N/A")
        print(f"❓ Question:\n{q_stem}\n")
        
        # Choices
        choices = rec.get("question", {}).get("choices", [])
        if choices:
            print("📋 Choices:")
            for c in choices:
                print(f"   [{c.get('label', '?')}] {c.get('text', '')}")
            print(f"   👉 Correct Answer Key: {rec.get('answerKey', 'N/A')}\n")
            
        # Concept
        clean_concept = rec.get("clean_scientific_concept") or "[MISSING CONCEPT]"
        print(f"💡 Scientific Concept:\n   {clean_concept}\n")
        
        # CoT Explanation
        explanation = rec.get("key_scientific_analogy") or "[MISSING EXPLANATION]"
        words = len(explanation.split())
        print(f"🧠 Extended Chain of Thought ({words} words):\n{explanation}\n")
        
        if is_def:
            print(f"⚠️ Defect Reason: {rec.get('defect_reason', 'Unknown')}\n")
            
        print(f"{'─'*70}\n")

# Default preview 5 samples
display_samples(num_samples=5, filter_by="all", random_sample=False)

📖 DISPLAYING 5 OF 448 SAMPLES (Filter: 'all', Random: False)

┌────────────────────────────────────────────────────────────────────┐
│ SAMPLE #01 | ID: rec0Y0PY1lx8aZPZh            | Status: ✅ SUCCESS   │
└────────────────────────────────────────────────────────────────────┘
❓ Question:
aniline is heated with sulfuric acid, forming product 1.

1 is treated with sodium bicarbonate, followed by sodium nitrite and HCl, forming product 2.

2 is allowed to react with 2-napthol, forming final product 3.

how many distinct nonexchaning hydrogen signals are there in the 1H nmr spectrum of 3?


📋 Choices:
   [A] 6
   [B] 7
   [C] 9
   [D] 8
   👉 Correct Answer Key: D

💡 Scientific Concept:
   Diazonium Coupling Reaction

🧠 Extended Chain of Thought (443 words):
The diazonium coupling reaction is a fundamental transformation in organic chemistry, specifically within the field of synthetic dye chemistry. At its core, it is a process that joins two distinct aromatic molecules together through a ni

## 9. 💾 Export Results to CSV
Converts the generated JSONL records into structured CSV files (both full and clean-only) inside the timestamped results directory.

In [81]:
def export_results_to_csv(
    jsonl_path: str = OUTPUT_DATASET_PATH,
    csv_path: str = OUTPUT_CSV_PATH,
    clean_csv_path: str = OUTPUT_CLEAN_CSV_PATH
) -> pd.DataFrame:
    """
    Exports the generated JSONL to both full CSV and clean (non-defective) CSV
    inside the timestamped results directory.
    """
    if not os.path.exists(jsonl_path):
        print(f"❌ Input JSONL file not found: {jsonl_path}")
        return pd.DataFrame()
        
    records = load_jsonl(jsonl_path)
    if not records:
        print("❌ No records found in JSONL file.")
        return pd.DataFrame()
        
    flattened_rows = []
    for r in records:
        choices_list = r.get("question", {}).get("choices", [])
        choices_formatted = " | ".join([f"{c.get('label')}: {c.get('text')}" for c in choices_list])
        
        row = {
            "id": r.get("id"),
            "question_stem": r.get("question", {}).get("stem"),
            "choices": choices_formatted,
            "answer_key": r.get("answerKey"),
            "scientific_concept": r.get("clean_scientific_concept"),
            "raw_concept_output": r.get("key_scientific_concept"),
            "chain_of_thought_explanation": r.get("key_scientific_analogy"),
            "is_defective": r.get("is_defective", False),
            "defect_reason": r.get("defect_reason"),
            "generation_status": r.get("generation_status", "unknown"),
            "teacher_model": r.get("generation_meta", {}).get("teacher_model", TEACHER_MODEL),
        }
        flattened_rows.append(row)
        
    df = pd.DataFrame(flattened_rows)
    
    # 1. Save Full CSV
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"✅ Full CSV Saved ({len(df)} records):")
    print(f"   📁 {csv_path}")
    
    # 2. Save Clean-only CSV (without defects)
    df_clean = df[~df["is_defective"]].copy()
    df_clean.to_csv(clean_csv_path, index=False, encoding="utf-8-sig")
    print(f"✅ Clean-Only CSV Saved ({len(df_clean)} non-defective records):")
    print(f"   📁 {clean_csv_path}\n")
    
    return df

# Run Export
df_results = export_results_to_csv()
if not df_results.empty:
    display(df_results.head(3))


✅ Full CSV Saved (448 records):
   📁 results/run_2026-08-11_10-37-04/GPQA_cot-600w_deepseek-v4-flash.csv
✅ Clean-Only CSV Saved (448 non-defective records):
   📁 results/run_2026-08-11_10-37-04/GPQA_cot-600w_deepseek-v4-flash_clean.csv



,id,question_stem,choices,answer_key,scientific_concept,raw_concept_output,chain_of_thought_explanation,is_defective,defect_reason,generation_status,teacher_model
0,rec0Y0PY1lx8aZPZh,"aniline is heated with sulfuric acid, forming ...",A: 6 | B: 7 | C: 9 | D: 8,D,Diazonium Coupling Reaction,"{""key_scientific_concept"": ""Diazonium Coupling...",The diazonium coupling reaction is a fundament...,False,None,success,deepseek-v4-flash
1,rec0VuKUjt1SZ7NYv,Consider the following metric:\n\nds^{2}=\frac...,A: +\infty | B: 0 | C: 4\pi\left(x^{2}+y^{2}\r...,A,Pseudosphere Area Formula,"{""key_scientific_concept"": ""Pseudosphere Area ...",The pseudosphere is a fundamental geometric su...,False,None,success,deepseek-v4-flash
2,rec055vn3qEqKHHTc,"A large gene has dozens of exons, of which the...",A: R-loops | B: polyA tail | C: lariat | D: an...,A,Antisense Oligonucleotide Exon Skipping,"{""key_scientific_concept"": ""Antisense Oligonuc...",Antisense oligonucleotide exon skipping is a t...,False,None,success,deepseek-v4-flash
